# 06. Bridge to the agent and RL phase

The second phase of the project uses decision models inside an agent. A decision model is fast and cheap, so it works well as a router or a guard: it chooses the next action, while deterministic code enforces hard rules and uncertain decisions are escalated.

We frame a small sequential task as an RL-style problem. The reward is task success minus a cost per step. The mock lets us simulate a learned policy without any download or key.


<img src='../images/agent_loop.png' alt='Agent loop' width='620'>

*The decision model scores actions, deterministic code enforces rules and gates, and the environment returns a new state plus a reward.*


## Step 1. Define the action set

Our toy agent has three actions. Declaring them explicitly is the same idea as a Choice question: the model can only pick from what we list.


In [ ]:
ACTIONS = ['retrieve', 'answer', 'escalate']
print(ACTIONS)
assert len(ACTIONS) == 3


## Step 2. A simulated learned policy

We make the mock prefer `retrieve` when there is no evidence and `answer` when evidence is present. This stands in for a model trained on real traces. The gold index is only a teaching device here.


In [ ]:
from decision_qa import MockClient, ChoiceQuestion
client = MockClient(seed=7)
state = {'question': 'What did the trial report?', 'has_evidence': False}
print(state)


In [ ]:
def policy_question(state):
    target = 0 if not state['has_evidence'] else 1
    return ChoiceQuestion(label='next_action', options=ACTIONS, gold_index=target)


In [ ]:
q0 = policy_question(state)
print(q0.options[q0.gold_index])


In [ ]:
res0 = client.decision(state, [q0])[0]
print(res0.probabilities)


In [ ]:
action0 = max(res0.probabilities, key=res0.probabilities.get)
print(action0)
assert action0 == 'retrieve'


## Step 3. Deterministic rules come first

A decision model should not be trusted with hard constraints. We add a rule that forbids answering without evidence. Rule code is cheap, testable, and always correct.


In [ ]:
def enforce_rules(state, action):
    if action == 'answer' and not state['has_evidence']:
        return 'retrieve'
    return action


In [ ]:
print(enforce_rules(state, 'answer'))
assert enforce_rules(state, 'answer') == 'retrieve'


In [ ]:
state['has_evidence'] = True
print(enforce_rules(state, 'answer'))
assert enforce_rules(state, 'answer') == 'answer'


## Step 4. The confidence gate

If the top probability is below a threshold, we escalate instead of acting. We create a mock without gold information to produce a low, uniform confidence.


In [ ]:
uniform_client = MockClient(seed=1, quality=0.0)
uncertain = ChoiceQuestion(label='next_action', options=ACTIONS)
res_u = uniform_client.decision(state, [uncertain])[0]
print(res_u.probabilities)


In [ ]:
def gate(result, threshold=0.6):
    action = max(result.probabilities, key=result.probabilities.get)
    confidence = result.probabilities[action]
    return 'escalate' if confidence < threshold else action


In [ ]:
print(gate(res_u))
assert gate(res_u) == 'escalate'


## Step 5. One agent step

A step combines the three pieces: ask the model, enforce rules, apply the gate. We write it as one small function and test it on a state without evidence.


In [ ]:
def agent_step(state):
    q = policy_question(state)
    result = client.decision(state, [q])[0]
    action = max(result.probabilities, key=result.probabilities.get)
    action = enforce_rules(state, action)
    action = gate(result) if result.probabilities[action] < 0.6 else action
    return action, result.probabilities[action]


In [ ]:
fresh = {'question': 'What did the trial report?', 'has_evidence': False}
action, conf = agent_step(fresh)
print(action, round(conf, 3))
assert action == 'retrieve'


## Step 6. Run one full episode

An episode repeats steps until the agent answers, escalates, or runs out of steps. Retrieving sets the evidence flag. Answering returns a reward reduced by the number of steps taken.


In [ ]:
def episode(max_steps=4):
    s = {'question': 'What did the trial report?', 'has_evidence': False}
    for step in range(1, max_steps + 1):
        action, _ = agent_step(s)
        if action == 'retrieve':
            s['has_evidence'] = True
        elif action == 'answer':
            return 1.0 - 0.1 * step
        else:
            return -0.2
    return -0.1


In [ ]:
reward = episode()
print(round(reward, 3))
assert -1.0 <= reward <= 1.0


In [ ]:
rewards = [episode() for _ in range(20)]
mean_reward = sum(rewards) / len(rewards)
print(round(mean_reward, 3))


Plotting the reward per episode shows whether the policy is stable. A flat line at 0.8 means the agent retrieves once and then answers, while negative spikes mark escalations. The dashed line is the mean reward across episodes.


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5.2, 2.6), constrained_layout=True)
ax.plot(range(1, len(rewards) + 1), rewards, 'o-', color='#54A24B', label='episode reward')
ax.axhline(mean_reward, color='#8A5A00', linestyle='--', label=f'mean {mean_reward:.2f}')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward')
ax.set_title('Reward per episode')
ax.set_ylim(-0.4, 1.0)
ax.grid(alpha=0.25)
ax.legend(loc='lower right', fontsize=8)
plt.show()


## Step 7. The RL vocabulary

The state is what the agent observes. The action is the typed Choice result. The policy is the decision model plus the rules and the gate. The reward scores the outcome. In a real Phase 2 task we would replace the mock with a local open model and define a real reward, for example exact match minus latency cost on a multi-hop QA benchmark.

Free projects to study next: JevLoop and ProgressGate, both open source and documented in the curated awesome-decision-models list. Every tool used in this repository is free.
